# Main model on Colab GPU (Multi-Scale PatchTST, with and without news)

**Before running:** Runtime → Change runtime type → GPU. With Colab Pro, **L4** is a good choice (faster than T4, far cheaper than A100). Then Runtime → Run all.

Inputs: the GitHub repo (cloned below) and the two raw price files in `MyDrive/Senior/prices/`.
Results are written straight to `MyDrive/Senior/results/` after **every seed**, so a disconnect loses at most one seed.
Everything is scored on the validation block only; the test block stays sealed. Keep this tab open while it runs.

In [ ]:
!nvidia-smi -L

In [ ]:
!git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
!grep -c DEVICE scripts/baselines_deep.py  # must be > 0 (GPU support)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PRICE_DIR = '/content/drive/MyDrive/Senior/prices'
OUT_DIR = '/content/drive/MyDrive/Senior/results'
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo "$OUT_DIR"
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/

In [ ]:
# Files must be byte-identical to the PC copies (hashes recorded 2026-10-15)
import hashlib
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}  # Git copy (LF); the PC copy is CRLF, sha c5c111f6…
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
!python tests/test_pipeline.py

In [ ]:
# 1) Main model WITH news (v1)
!python scripts/baselines_deep.py --models multiscale_sector --news data/derived/sentiment_scores.csv --out "$OUT_DIR/main_model_v1_news.json"

In [ ]:
# 2) Main model WITHOUT news, rerun on the same GPU so the comparison is like for like
!python scripts/baselines_deep.py --models multiscale_sector --out "$OUT_DIR/main_model_v0_gpu.json"

In [ ]:
# 3) Multi-scale ablation partner: one scale, same sector label, with news
!python scripts/baselines_deep.py --models singlescale_sector --news data/derived/sentiment_scores.csv --out "$OUT_DIR/singlescale_v1_news.json"

In [ ]:
# Summary (validation macro-F1, mean over seeds)
import json, os, statistics as st
for f in ['main_model_v1_news.json', 'main_model_v0_gpu.json', 'singlescale_v1_news.json']:
    p = os.path.join(OUT_DIR, f)
    if not os.path.exists(p):
        print(f, 'missing'); continue
    r = json.load(open(p))
    for name, m in r['models'].items():
        runs = m['runs']
        print(f, name, r.get('device'), r.get('gpu'), 'seeds', len(runs),
              {h: round(st.mean(x['val'][h]['macro_f1'] for x in runs), 3) for h in ('h1', 'h5', 'h10')})